In [ ]:
import pandas as pd

train = pd.read_csv("../data/UNSW_NB15_testing-set.csv")
test = pd.read_csv("../data/UNSW_NB15_training-set.csv")

print(train.shape, test.shape)

(175341, 45) (82332, 45)


In [3]:
train.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [4]:
train['label'].value_counts()

label
1    119341
0     56000
Name: count, dtype: int64

In [5]:
train['attack_cat'].value_counts()

attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64

In [7]:
train.isna().sum()

id                   0
dur                  0
proto                0
service              0
state                0
spkts                0
dpkts                0
sbytes               0
dbytes               0
rate                 0
sttl                 0
dttl                 0
sload                0
dload                0
sloss                0
dloss                0
sinpkt               0
dinpkt               0
sjit                 0
djit                 0
swin                 0
stcpb                0
dtcpb                0
dwin                 0
tcprtt               0
synack               0
ackdat               0
smean                0
dmean                0
trans_depth          0
response_body_len    0
ct_srv_src           0
ct_state_ttl         0
ct_dst_ltm           0
ct_src_dport_ltm     0
ct_dst_sport_ltm     0
ct_dst_src_ltm       0
is_ftp_login         0
ct_ftp_cmd           0
ct_flw_http_mthd     0
ct_src_ltm           0
ct_srv_dst           0
is_sm_ips_ports      0
attack_cat 

In [8]:
train.dtypes

id                     int64
dur                  float64
proto                    str
service                  str
state                    str
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct_src_dport_l

In [9]:
features = pd.read_csv("../data/NUSW-NB15_features.csv", encoding="latin-1")
features

,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol
5,6,state,nominal,Indicates to the state and its dependent proto...
6,7,dur,Float,Record total duration
7,8,sbytes,Integer,Source to destination transaction bytes
8,9,dbytes,Integer,Destination to source transaction bytes
9,10,sttl,Integer,Source to destination time to live value


In [10]:
train.drop(columns='id').duplicated().sum()

np.int64(67601)

In [11]:
dupes = train.drop(columns='id').duplicated()
train[dupes]['label'].value_counts()

label
1    63491
0     4110
Name: count, dtype: int64

In [12]:
train[dupes]

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
66,67,50.004398,ospf,-,INT,6,0,384,0,0.099991,...,2,1,0,0,0,1,2,0,Normal,0
169,170,0.000011,udp,-,INT,2,0,2120,0,90909.090200,...,2,63,0,0,0,8,62,0,Normal,0
214,215,0.001146,udp,dns,CON,2,2,146,178,2617.801058,...,2,3,0,0,0,2,8,0,Normal,0
313,314,0.001098,udp,dns,CON,2,2,132,164,2732.240490,...,2,3,0,0,0,3,12,0,Normal,0
347,348,0.001088,udp,dns,CON,2,2,130,162,2757.352893,...,2,2,0,0,0,2,2,0,Normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175335,175336,0.000006,udp,dns,INT,2,0,114,0,166666.660800,...,17,45,0,0,0,33,45,0,Generic,1
175336,175337,0.000009,udp,dns,INT,2,0,114,0,111111.107200,...,13,24,0,0,0,24,24,0,Generic,1
175338,175339,0.000009,udp,dns,INT,2,0,114,0,111111.107200,...,3,13,0,0,0,3,12,0,Generic,1
175339,175340,0.000009,udp,dns,INT,2,0,114,0,111111.107200,...,14,30,0,0,0,30,30,0,Generic,1


In [13]:
train[['proto', 'service', 'state']].nunique()

proto      133
service     13
state        9
dtype: int64

In [14]:
train['proto'].value_counts().head(10)

proto
tcp       79946
udp       63283
unas      12084
arp        2859
ospf       2595
sctp       1150
any         300
gre         225
sun-nd      201
swipe       201
Name: count, dtype: int64

## Exploration summary

- Data file names swapped (testing-set.csv has 175,341 rows which matches training set size and vice versa)

- 119,341 attacks vs 56,000 normal (the model learns from regular traffic so it gets 56,000 rows to train from, 68/32 split is unrealistic for normal networks)

- 9 different attacks with uneven counts (40,000 generic but only 130 worms, when testing check results for each type separately, rarer types could be missed without affecting overall score)

- No missing values (service uses "-" for unknown which .isna() doesn't catch, this is fine for this case and will be kept)

- 67,601 duplicate rows which is 39% (63,491 attacks vs 4,110 normal) normal duplicates will be removed so the model doesn’t over-learn them

- Proto, service and state are text and will need converting to numbers

- Proto has 133 values mainly tcp and udp (rarer protocols will be grouped as "other")

  
## Preprocessing plan

- Drop "id"
- Remove "label" and "attack_cat" from model input
- Train on normal rows
- Remove normal dupes
- Group rare protocols as "other"
- Convert text columns to numbers   